# Osaka test — 1 hexagon, 1 keyword, 1 API call
API key goes in `.env`. Kernel: `.venv`.

In [ ]:
import ast
import os
from pathlib import Path

import folium
import geopandas as gpd
import h3
import pandas as pd
from dotenv import load_dotenv
from shapely.geometry import Polygon

from modules.pipeline.data_collection_pipeline import DataCollectionPipeline, current_date
from run_collection import FIELD_MASK

KEYWORD = "restaurant"
LAT_LNG = (34.7025, 135.4959)  # Umeda; tests the hexagon containing this point
MAX_API_CALLS = 1

load_dotenv(override=True)
api_key = os.environ.get("GOOGLE_MAPS_API_KEY")
assert api_key, "Put GOOGLE_MAPS_API_KEY in .env"

pipeline = DataCollectionPipeline(city_name="Osaka", api_key=api_key, field_mask=FIELD_MASK, output_dir="output_test")
pipeline.text_query = [KEYWORD]
pipeline.max_resolution = 0  # no splitting into child hexagons

# Hard cap on API calls, and only 1 page (20 places) per search
_search, api_calls = pipeline.google_maps_api.text_search_api, []
def guarded_search(**kwargs):
    if len(api_calls) >= MAX_API_CALLS:
        raise RuntimeError(f"Blocked: MAX_API_CALLS={MAX_API_CALLS} reached")
    api_calls.append(kwargs["text_query"])
    results = _search(**kwargs)
    if isinstance(results, dict):
        results.pop("nextPageToken", None)
    return results
pipeline.google_maps_api.text_search_api = guarded_search

In [ ]:
pipeline.collect_country_shp_files()
all_hexagons = pipeline.generate_hexagons()
test_hex = h3.geo_to_h3(*LAT_LNG, h3.h3_get_resolution(next(iter(all_hexagons))))
assert test_hex in all_hexagons
pipeline.hexagons = {test_hex}
print("Test hexagon:", test_hex)

Generated 21 hexagons for Osaka.
Test hexagon: 862e61057ffffff


In [ ]:
osaka = pipeline.gdf[pipeline.gdf[f"NAME_{pipeline.level}"] == "Osaka"][[f"NAME_{pipeline.level}", "geometry"]]
hex_gdf = gpd.GeoDataFrame(
    {"hex": list(all_hexagons)},
    geometry=[Polygon(h3.h3_to_geo_boundary(h, geo_json=True)) for h in all_hexagons],
    crs="EPSG:4326",
)
test_hex_gdf = hex_gdf[hex_gdf["hex"] == test_hex]

MAP_DIR = Path("output_test/maps")
MAP_DIR.mkdir(parents=True, exist_ok=True)

def base_map():
    return osaka.explore(tiles="https://cyberjapandata.gsi.go.jp/xyz/pale/{z}/{x}/{y}.png", attr="地理院タイル", color="#333", style_kwds={"fillOpacity": 0.05, "weight": 2}, name="Osaka")

def save(m, name):
    folium.LayerControl().add_to(m)
    path = MAP_DIR / f"{name}.html"
    m.save(path)
    print("Saved:", path.resolve())
    return m

# 1. Osaka City
save(base_map(), "1_osaka")

Saved: /Users/driandy/Desktop/Works/Artefact/001 - SMBC Japan/output_test/maps/1_osaka.html


In [ ]:
# 2. Osaka + hexagons (test hexagon in orange)
m = base_map()
hex_gdf.explore(m=m, color="#2b6cb0", style_kwds={"fillOpacity": 0.05}, name="Hexagons")
test_hex_gdf.explore(m=m, color="#dd6b20", style_kwds={"fillOpacity": 0.4}, name="Test hexagon")
save(m, "2_osaka_hexagons")

Saved: /Users/driandy/Desktop/Works/Artefact/001 - SMBC Japan/output_test/maps/2_osaka_hexagons.html


Next cell spends **1 API call**.

In [ ]:
pipeline.get_google_maps_data_extraction()
pipeline.combine_all_raw_data()
print("API calls used:", len(api_calls))

Retrieving Google Maps doors for keyword: restaurant
Extracting doors Osaka for hex_id: 862e61057ffffff for text_query: restaurant.
862e61057ffffff returns 20 doors for keyword: restaurant
total dataframe size:  20
API calls used: 1


In [ ]:
df = pd.read_csv(Path("output_test/google-data-extraction") / current_date / "combined/data-extraction.csv")
parse = lambda v: ast.literal_eval(v) if isinstance(v, str) else {}
df["name"] = df["displayName"].map(lambda v: parse(v).get("text"))
print(len(df), "places")
df[["name", "formattedAddress", "primaryType", "businessStatus", "googleMapsUri"]]

20 places


,name,formattedAddress,primaryType,businessStatus,googleMapsUri
0,誠 cuisine,〒541-0058 大阪府大阪市中央区南久宝寺町２丁目１−１１ 中船場スクエアビル 1F,restaurant,OPERATIONAL,https://maps.google.com/?cid=91141369485581511...
1,ル ビストロ クードポール,〒540-0036 大阪府大阪市中央区船越町１丁目３−５,french_restaurant,OPERATIONAL,https://maps.google.com/?cid=41788382043476130...
2,Restaurant つじ川,〒542-0081 大阪府大阪市中央区南船場３丁目２−２８,restaurant,OPERATIONAL,https://maps.google.com/?cid=99485557011494778...
3,地中海の恵みと和食ごはんひなた,〒540-0008 大阪府大阪市中央区大手前３丁目１−６９,japanese_restaurant,OPERATIONAL,https://maps.google.com/?cid=16775759077411199...
4,STYLISH GRILL Tyu,〒541-0059 大阪府大阪市中央区博労町３丁目３−１３,japanese_restaurant,OPERATIONAL,https://maps.google.com/?cid=17924658497954121...
5,CHELSEA,〒540-0036 大阪府大阪市中央区徳井町１丁目３−２ ホテルマイステイズ大手前 1階,italian_restaurant,OPERATIONAL,https://maps.google.com/?cid=68381500254380491...
6,VINOTECA Osaka,〒541-0059 大阪府大阪市中央区博労町１丁目８−１２,restaurant,OPERATIONAL,https://maps.google.com/?cid=15516176557705304...
7,Restaurant RiRe,〒540-0004 大阪府大阪市中央区玉造２丁目４−１３ 上町ロングピア 1F,french_restaurant,OPERATIONAL,https://maps.google.com/?cid=16267237489380706...
8,HINOKA,〒540-0037 大阪府大阪市中央区内平野町１丁目１−６ 1階,italian_restaurant,OPERATIONAL,https://maps.google.com/?cid=15332396959380252...
9,キッチン&バーSOL,〒541-0043 大阪府大阪市中央区高麗橋２丁目４−１３ 串ダイニングＳＯＬ,japanese_izakaya_restaurant,OPERATIONAL,https://maps.google.com/?cid=52911175831589562...


In [ ]:
# 3. Osaka + test hexagon + merchants
df["lat"] = df["location"].map(lambda v: parse(v).get("latitude"))
df["lng"] = df["location"].map(lambda v: parse(v).get("longitude"))
merchants = gpd.GeoDataFrame(
    df[["name", "primaryType", "formattedAddress", "businessStatus", "googleMapsUri"]],
    geometry=gpd.points_from_xy(df["lng"], df["lat"]),
    crs="EPSG:4326",
)

m = base_map()
test_hex_gdf.explore(m=m, color="#dd6b20", style_kwds={"fillOpacity": 0.15}, name="Test hexagon")
merchants.explore(m=m, color="#c53030", marker_type="circle_marker", marker_kwds={"radius": 6}, name=f"Merchants ({len(merchants)})")
min_lng, min_lat, max_lng, max_lat = test_hex_gdf.total_bounds
m.fit_bounds([[min_lat, min_lng], [max_lat, max_lng]])
save(m, "3_osaka_hexagon_merchants")

Saved: /Users/driandy/Desktop/Works/Artefact/001 - SMBC Japan/output_test/maps/3_osaka_hexagon_merchants.html
